# Predicción de Ingresos — Census Income KDD

Pipeline modular: este notebook orquesta los pasos del proyecto importando desde el paquete `census_ml` (en `src/`). Toda la implementación detallada vive en los módulos; aquí solo encadenamos las llamadas y discutimos los resultados.

Para reproducir el pipeline completo desde la línea de comandos:

```bash
uv run python scripts/train_all.py        # entrena los 4 modelos y guarda los .pkl
uv run python scripts/evaluate.py         # tabla comparativa + ensambles
```


## 0. Setup

Agregamos `src/` al `sys.path` para poder importar `census_ml` sin instalar el paquete.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Carga del dataset

El dataset Census Income KDD (UCI id=117) se descarga dinámicamente vía `ucimlrepo`. Tiene 199.523 registros, 41 variables predictoras y un fuerte desbalance (6.3% positivos). Ver detalles en [INFORME_PROYECTO.md](../INFORME_PROYECTO.md).

In [ ]:
from census_ml.data import load_census_data, split_train_test

X, y = load_census_data()
print(f'Shape: {X.shape}  |  Distribución target: {y.value_counts(normalize=True).to_dict()}')


## 2. División train/test (80/20, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(X, y)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train positivos: {y_train.mean():.4f}  |  Test positivos: {y_test.mean():.4f}')


## 3. Preprocesamiento (5 etapas)

`preprocess_full` aplica las 5 etapas en orden:

1. **Imputación** — mediana (numéricas) / moda (categóricas)
2. **Encoding alta cardinalidad** — `OrdinalEncoder` para `AHGA` (educación, jerárquica) + Target Encoding para el resto
3. **Encoding baja cardinalidad** — `OneHotEncoder`
4. **Escalado** — `StandardScaler`
5. **Balanceo** — `SMOTE(k_neighbors=5)` solo en train

Todos los estadísticos se ajustan SOLO sobre train (sin data leakage).

In [ ]:
from census_ml.preprocessing import preprocess_full

X_train_res, y_train_res, X_test_proc, artifacts = preprocess_full(X_train, X_test, y_train)
print(f'Train balanceado: {X_train_res.shape}  (50/50 post-SMOTE)')
print(f'Test preprocesado: {X_test_proc.shape}  (conserva 6.3% positivos)')
print(f'\nFeatures generadas:')
print(f'  Numéricas:                {len(artifacts.num_cols)}')
print(f'  One-Hot expandidas:       {len(artifacts.one_hot_cols)}')
print(f'  Target encoded:           {len(artifacts.nominal_high_cols)}')


## 4. Entrenamiento de los 4 modelos

> **Nota**: re-entrenar todo (especialmente LightGBM con Optuna 100 trials) toma horas. Si los modelos ya están serializados en `models/`, salta a la sección 5 y carga directamente desde disco.

### 4.1 Logistic Regression ElasticNet

In [ ]:
from census_ml.models import train_logistic_elasticnet

# Descomentar para reentrenar:
# logreg = train_logistic_elasticnet(X_train_res, y_train_res)
# joblib.dump(logreg, ROOT / 'models/logit_elasticnet.pkl')


### 4.2 Random Forest tuneado (RandomizedSearchCV + threshold tuning)

In [ ]:
from census_ml.models import train_random_forest_tuned, tune_threshold_f1

# Descomentar para reentrenar:
# rf, best_params = train_random_forest_tuned(X_train_res, y_train_res)
# joblib.dump(rf, ROOT / 'models/random_forest_tunned.pkl')
# print(f'Mejores parámetros: {best_params}')


### 4.3 XGBoost (configuración fija: 600 árboles, depth=8, lr=0.05)

In [ ]:
from census_ml.models import train_xgboost

# Descomentar para reentrenar:
# xgb = train_xgboost(X_train_res, y_train_res)
# joblib.dump(xgb, ROOT / 'models/xgb_model.pkl')


### 4.4 LightGBM tuneado con Optuna (100 trials, 5-fold CV, métrica AUC-PR)

In [ ]:
from census_ml.models import train_lightgbm_tuned

# ATENCIÓN: este paso toma varias horas. Descomentar solo si quieres recorrer Optuna:
# lgbm, study = train_lightgbm_tuned(X_train_res, y_train_res, n_trials=100)
# joblib.dump(lgbm, ROOT / 'models/lgbm_tuned.pkl')
# joblib.dump(study, ROOT / 'models/lgbm_study.pkl')
# print(f'Mejor AUC-PR (CV): {study.best_value:.4f}')


## 5. Carga de modelos serializados y predicciones

In [ ]:
from census_ml.config import MODEL_FILES, MODELS_DIR

models = {name: joblib.load(MODELS_DIR / fname) for name, fname in MODEL_FILES.items()}

predictions = {}
for name, model in models.items():
    y_prob = model.predict_proba(X_test_proc)[:, 1]
    y_pred = model.predict(X_test_proc)
    predictions[name] = {'y_prob': y_prob, 'y_pred': y_pred}
    print(f'  {name:<28} -> {len(y_pred):,} predicciones listas')


## 6. Tabla comparativa de modelos

In [ ]:
from census_ml.evaluation import comparison_table, METRIC_COLS

individual = comparison_table(predictions, y_test.values)
individual.style.background_gradient(cmap='Greens', axis=0).format('{:.4f}')


### 6.1 Curvas ROC y Precision-Recall superpuestas

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

colors = ['#1D9E75', '#D4537E', '#3B7DD8', '#F39C12']
fig, axes = plt.subplots(2, 1, figsize=(10, 12))

for (name, p), color in zip(predictions.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, p['y_prob'])
    auc = roc_auc_score(y_test, p['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2.2, label=f'{name}  ({auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curvas ROC', fontweight='bold'); axes[0].legend(loc='lower right')

baseline = float(y_test.mean())
for (name, p), color in zip(predictions.items(), colors):
    prec_c, rec_c, _ = precision_recall_curve(y_test, p['y_prob'])
    ap = average_precision_score(y_test, p['y_prob'])
    axes[1].plot(rec_c, prec_c, color=color, lw=2.2, label=f'{name}  ({ap:.3f})')
axes[1].axhline(baseline, color='gray', linestyle='--', lw=1.5, label=f'Baseline ({baseline:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curvas Precision-Recall', fontweight='bold'); axes[1].legend(loc='upper right')
plt.tight_layout(); plt.show()


## 7. Ensambles

Combinamos las probabilidades de los 4 modelos individuales:
- **Soft Voting**: promedio aritmético simple (peso 1/N para cada modelo).
- **Weighted Average**: ponderación proporcional al AUC-PR de cada modelo en test.

In [ ]:
from census_ml.ensemble import soft_voting, weighted_average_by_aucpr
from census_ml.evaluation import compute_metrics

# Soft Voting
soft_prob, soft_pred = soft_voting(predictions)
soft_metrics = compute_metrics(y_test.values, soft_pred, soft_prob)

# Weighted Average por AUC-PR
w_prob, w_pred, weights = weighted_average_by_aucpr(predictions, individual['AUC-PR'])
w_metrics = compute_metrics(y_test.values, w_pred, w_prob)

print('Pesos del Weighted Average:')
for name, w in weights.items():
    print(f"  {name:<25}  AUC-PR={individual.loc[name, 'AUC-PR']:.4f}  ->  peso={w:.4f}")


### 7.1 Tabla final: individuales vs ensambles

In [ ]:
final = individual.copy()
final.loc['Ensemble — Soft Voting'] = soft_metrics
final.loc['Ensemble — Weighted Average'] = w_metrics

final.style.background_gradient(cmap='Greens', axis=0).format('{:.4f}')


## 8. Conclusiones

- **XGBoost** (sin tuning explícito) es el mejor modelo individual en F1 y AUC-PR.
- **LightGBM tuneado** alcanza la mayor precisión, a costa de recall.
- **Logistic Regression** es el más permisivo (recall ≈ 0.87).
- El **Weighted Average ponderado por AUC-PR** es la mejor estrategia general — supera a cada modelo individual al combinar la diversidad estructural de los 4.

Para la justificación detallada de cada decisión técnica, ver [INFORME_PROYECTO.md](../INFORME_PROYECTO.md).